### Exploratory Data Analysis

In [40]:
import pandas as pd
import plotly.express as px
import nbformat
import plotly.graph_objects as go
import datetime 
import seaborn as sns

from plotly.subplots import make_subplots

### Functions

In [37]:
def get_merged_data(electric_df: pd.DataFrame, hot_water_df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns merged electric & hot water dataframe for corresponding year
    """
    electric_df = electric_df.melt(
        id_vars=['ts'], 
        var_name='building_name', 
        value_name='electrical_energy'
    )

    hot_water_df = hot_water_df.melt(
        id_vars=['ts'],
        var_name='building_name',
        value_name='hot_water_energy'
    )
    
    merged_df = pd.merge(electric_df,
                        hot_water_df,
                        on=['ts', 'building_name'],
                        how='outer')
    
    merged_df['ts'] = merged_df['ts'].str.replace(' Rel', '', regex=False)
        
    merged_df = (
        merged_df
        .convert_dtypes()                 # switch to nullable dtypes (Float64, Int64)
        .where(pd.notna(merged_df), None) # replace missing with None
    )

    merged_df = merged_df.set_index('ts')

    return merged_df

def get_yearly_data(year) -> pd.DataFrame:
    """
    Returns electric & hot water dataframe for specified year
    """
    yearly_elec_path = f"../data/processed/electrical-energy/electrical_energy_{year}.csv"
    yearly_hot_water_path = f"../data/processed/hot-water-energy/hot_water_energy_{year}.csv"
        
    return get_merged_data(pd.read_csv(yearly_elec_path), pd.read_csv(yearly_hot_water_path))

def get_building_monthly_info(old_df: pd.DataFrame, building_name:str) -> pd.DataFrame:
    df = old_df[old_df['building_name'] == building_name]

    df.index = pd.to_datetime(
        df.index.str.split(" ").str[0].str.replace("Z", "", regex=False),
        format="%Y-%m-%dT%H:%M:%S"
    )

    df.drop('building_name', axis = 1, inplace=True)

    df = df.resample('MS').mean().reset_index()
    
    return df

def get_building_weekly_info(old_df: pd.DataFrame, building_name:str, year:int, week:int) -> pd.DataFrame:

    d = f"{year}-W{week:02d}"
    week_start = datetime.datetime.strptime(d + '-1', "%G-W%V-%u")
    week_end = week_start + datetime.timedelta(days = 7)

    df = old_df[old_df['building_name'] == building_name]

    df.index = pd.to_datetime(
        df.index.str.split(" ").str[0].str.replace("Z", "", regex=False),
        format="%Y-%m-%dT%H:%M:%S"
    )

    df.drop('building_name', axis = 1, inplace=True)

    df = df[(df.index >= week_start) & (df.index < week_end)]
        
    return df.reset_index()

def get_building_daily_info(old_df: pd.DataFrame, building_name:str, year:int, month:int, day:int) -> pd.DataFrame:

    day_start = datetime.datetime(year, month, day, 0, 0, 0)
    day_end = day_start + datetime.timedelta(hours = 24)

    df = old_df[old_df['building_name'] == building_name]

    df.index = pd.to_datetime(
        df.index.str.split(" ").str[0].str.replace("Z", "", regex=False),
        format="%Y-%m-%dT%H:%M:%S"
    )

    df.drop('building_name', axis = 1, inplace = True)

    df = df[(df.index >= day_start) & (df.index < day_end)]

    return df.reset_index()

def create_yearly_heatmap(df, year, start_day, start_month, end_day, end_month, term):
    
    start_date = datetime.datetime(year, start_month, start_day, 0, 0, 0)
    end_date = datetime.datetime(year, end_month, end_day, 0, 0, 0)

    df = df[(df.index > start_date) & (df.index < end_date)]

    print(df.shape)

    corr = df[['electrical_energy', 'hot_water_energy', 'temperature_2m', 
                'wind_speed_10m', 'precipitation', 
                'relative_humidity_2m', 'cloud_cover', 'apparent_temperature']].corr()

    fig = px.imshow(corr, 
                    text_auto='.2f',        # shows correlation values
                    color_continuous_scale='RdBu_r',  # red-blue diverging scale
                    zmin=-1, zmax=1,        # correlation range
                    title=f"Weather vs Energy Correlation Heatmap ({year} {term})")
    
    return fig

### Are Building Energy Features Good Indicators of Occupancy?

We collected electrical energy and hot water usage data from Skyspark spanning 2021–2025 (inclusive). To investigate whether energy usage patterns are associated with building occupancy, we examined monthly, weekly, and daily trends for several highly frequented campus buildings, including the AMS Nest, I.K. Barber Learning Centre, and ICICS. Our exploratory analysis primarily focuses on data from 2023, as energy usage patterns from earlier years were likely influenced by disruptions related to the COVID-19 pandemic.

Under the assumption that energy usage is associated with occupancy levels, we expect monthly energy usage to peak during Winter Term 2 of 2023 and Winter Term 1 of 2024, when campus activity is expected to be relatively high. For weekly trends, we expect energy usage to be consistently higher on weekdays than weekends due to increased campus activity during instructional days. Lastly, for daily trends, we expect energy usage to increase during daytime hours and decrease later in the evening as building occupancy declines.

In [ ]:
skyspark_2023 = get_yearly_data(2023)

entries = []

for building in ["AMS Nest", "I.K. Barber", "ICICS"]:
    df = get_building_monthly_info(skyspark_2023, building)
    df["building"] = building
    entries.append(df)

monthly_elec_usage = pd.concat(entries)

#### Electrical Energy Usage

##### Monthly Electrical Energy Usage in 2023

In [29]:
fig_01 = px.line(monthly_elec_usage, 
        x='ts', 
        y='electrical_energy',
        color='building',
        markers=True,
        title = "Monthly Electrical Energy Usage in 2023",
        labels=dict(ts = "Month", 
                    electrical_energy = "Electrical Energy (kwh)",
                    building = "Building Name"))

semesters = [
    ("2023 WT2", "2023-01-05", "2023-04-27", "rgba(100, 149, 237, 0.2)"),
    ("2023 ST1", "2023-05-10", "2023-06-17", "rgba(255, 165, 0, 0.2)"),
    ("2023 ST2", "2023-07-05", "2023-08-12", "rgba(255, 200, 0, 0.2)"),
    ("2024 WT1", "2023-09-08", "2023-12-22", "rgba(100, 149, 237, 0.2)"),
]

for name, start, end, color in semesters:
    fig_01.add_vrect(x0=start, x1=end, fillcolor=color, line_width=0,
                  annotation_text=name, annotation_position="top left")
    
fig_01.show()

We initially hypothesized that electrical energy usage would be higher during the winter academic terms (2023 WT2 and 2024 WT1), as student enrollment and campus activity are generally greater during these periods compared to the summer terms (2023 ST1 and 2023 ST2). However, the observed trends contradict this expectation. Across all three buildings (IKB, Nest, and ICICS), electrical energy usage was highest during the summer months despite lower expected occupancy levels. As expected, ICICS consistently consumed less electrical energy than both the Nest and IKB, likely due to its comparatively smaller building size.

One possible explanation is that a substantial portion of electrical energy consumption is driven by building operations, such as air conditioning and climate control systems, rather than direct student activity alone. Nevertheless, these findings do not necessarily imply that electrical energy usage is a poor indicator of occupancy. Instead, the relationship between occupancy and electrical usage may be obscured when analyzing aggregated annual trends. Examining electrical usage at finer temporal resolutions, such as weekly or daily intervals, may reveal occupancy-related patterns more clearly. Additionally, unaccounted temporal factors, including season and month, may help explain why the annual trends differed from our original expectations.

##### Weekly Electrical Energy Usage in 2023 (Week 02)

In [ ]:
entries = []

for building in ["AMS Nest", "I.K. Barber", "ICICS"]:
    df = get_building_weekly_info(skyspark_2023, building, 2023, 2)
    df["building"] = building
    entries.append(df)

weekly_elec_usage = pd.concat(entries)

In [27]:
fig_02 = px.line(weekly_elec_usage, 
        x='ts', 
        y='electrical_energy',
        color='building',
        markers=False,
        title = "Weekly Electrical Energy Usage in 2023-W02",
        labels=dict(ts = "Day", 
                    electrical_energy = "Electrical Energy (kwh)",
                    building = "Building Name"))

boxes = [
    ("Weekday", "2023-01-09", "2023-01-14", "rgba(100, 149, 237, 0.2)"),
    ("Weekend", "2023-01-14", "2023-01-16", "rgba(255, 165, 0, 0.2)"),
]

for name, start, end, color in boxes:
    fig_02.add_vrect(x0=start, x1=end, fillcolor=color, line_width=0,
                  annotation_text=name, annotation_position="top left")

fig_02.show()

In this visualization, the majority of peaks in electrical energy usage for both buildings occur during weekdays rather than weekends. This aligns with the expectation that campus facilities experience greater occupancy during instructional days, providing some evidence that electrical energy usage may be associated with building occupancy. However, this relationship cannot be confirmed from weekly trends alone. To investigate this relationship further, we examine electrical usage patterns at the daily level.

Note: Week 2 of 2023 was selected because it represents a period in which campus attendance and student activity are expected to be relatively high.

In [24]:
entries = []

for building in ["AMS Nest", "I.K. Barber", "ICICS"]:
    df = get_building_daily_info(skyspark_2023, building, 2023, 1, 13)
    df["building"] = building
    entries.append(df)

daily_elec_usage = pd.concat(entries)

##### Electrical Energy Usage in January 13th, 2023

In [25]:
fig_03 = px.line(daily_elec_usage, 
        x='ts', 
        y='electrical_energy',
        color='building',
        markers=False,
        title = "Electrical Energy Usage in January 13th, 2023 ",
        labels=dict(ts = "Day", 
                    electrical_energy = "Electrical Energy (kwh)",
                    building = "Building Name"))

fig_03.show()

Presented here is the electrical energy usage across a single day. Based on the visualization, electrical energy usage begins to increase between 05:00 and 06:00 and then stabilizes around a relatively consistent threshold for each building until the evening hours (approximately 19:00–20:00). The smaller but noticeable fluctuations observed between 09:00 and 18:00 may reflect the influence of student occupancy and daytime building activity on electrical energy consumption.

However, these findings should be interpreted with caution. We do not possess detailed information regarding how UBC facilities are operated and maintained, meaning that other factors, such as automated building systems, HVAC operations, scheduled maintenance, or institutional activities, may also contribute to the observed energy usage patterns.

#### Hot Water Energy Usage

In [30]:
fig_04 = px.line(monthly_elec_usage, 
        x='ts', 
        y='hot_water_energy',
        color = 'building', 
        markers=True, 
        title = "Monthly Hot Water Usage in 2023",
        labels=dict(ts = "Month", hot_water_energy = "Hot Water Energy (kwh)", building = "Building Name"))

semesters = [
    ("2023 WT2", "2023-01-05", "2023-04-27", "rgba(100, 149, 237, 0.2)"),
    ("2023 ST1", "2023-05-10", "2023-06-17", "rgba(255, 165, 0, 0.2)"),
    ("2023 ST2", "2023-07-05", "2023-08-12", "rgba(255, 200, 0, 0.2)"),
    ("2024 WT1", "2023-09-08", "2023-12-22", "rgba(100, 149, 237, 0.2)"),
]

for name, start, end, color in semesters:
    fig_04.add_vrect(x0=start, x1=end, fillcolor=color, line_width=0,
                  annotation_text=name, annotation_position="top left")

fig_04.update_layout(yaxis_range=[0, 500])

fig_04.show()

Contrary to the electrical energy usage data, hot water usage trends more closely align with our original hypothesis. Across the observed buildings, hot water usage tends to be higher during the winter academic terms than during the summer terms, when campus occupancy is expected to be lower. This may suggest that hot water consumption is more directly associated with human activity within buildings than electrical energy usage, which can also be heavily influenced by building operations such as climate control systems.

Nevertheless, temporal factors discussed previously, including seasonal and monthly effects, may still influence the observed hot water usage patterns and complicate their interpretation.

Additionally, an unusually large spike in hot water usage was observed for the Nest building in May 2023, reaching approximately 14,507.88 kWh. Since this value deviates substantially from surrounding observations, it may represent an outlier or an irregular operational event rather than a typical occupancy-related pattern.

In [31]:
fig_05 = px.line(weekly_elec_usage, 
        x='ts', 
        y='hot_water_energy',
        color='building',
        markers=False,
        title = "Weekly Hot Water Usage in 2023-W02",
        labels=dict(ts = "Day", 
                    hot_water_energy = "Hot Water Energy (kwh)",
                    building = "Building Name"))

boxes = [
    ("Weekday", "2023-01-09", "2023-01-14", "rgba(100, 149, 237, 0.2)"),
    ("Weekend", "2023-01-14", "2023-01-16", "rgba(255, 165, 0, 0.2)"),
]

for name, start, end, color in boxes:
    fig_05.add_vrect(x0=start, x1=end, fillcolor=color, line_width=0,
                  annotation_text=name, annotation_position="top left")

fig_05.show()

In [36]:
fig_06 = px.line(daily_elec_usage, 
        x='ts', 
        y='hot_water_energy',
        color='building',
        markers=False,
        title = "Hot Water Usage in January 13th, 2023 ",
        labels=dict(ts = "Day", 
                    hot_water_energy = "Electrical Energy (kwh)",
                    building = "Building Name"))

fig_06.show()

### Does Weather affect Energy Usage?

Weather can play a huge role when it comes to building occupancy. If the weather is too cold, students are more likely to head indoors, inadvertently using more energy. Alternatively, if the weather is hotter than normal, students are more likely to head outside (using less energy). To see whether or not weather truly has an effect on electrical usage and hot water usage, correlation matrices have been generated for the AMS Nest building to verify these correlations. 

In [12]:
weather_2023 = pd.read_csv("../data/processed/weather/weather_2023.csv", index_col=0)

weather_2023 = weather_2023.rename_axis("ts")

weather_2023.index = pd.to_datetime(
    weather_2023.index.str.replace("+00:00", "", regex=False))

weather_2023.head()

,temperature_2m,wind_speed_10m,snow_depth,is_day,precipitation,relative_humidity_2m,cloud_cover,apparent_temperature
ts,,,,,,,,
2023-01-01 00:00:00,7.30,9.178235,0.0,1.0,0.0,87.0,12.0,4.803664
2023-01-01 01:00:00,7.15,2.902413,0.0,0.0,0.0,88.0,0.0,5.572490
2023-01-01 02:00:00,6.95,7.993297,0.0,0.0,0.0,94.0,8.0,4.796128
2023-01-01 03:00:00,6.90,13.217443,0.0,0.0,0.0,93.0,2.0,3.938524
2023-01-01 04:00:00,6.80,13.779114,0.0,0.0,0.0,89.0,4.0,3.597236


In [13]:
ams_nest_2023 = skyspark_2023[skyspark_2023["building_name"] == "AMS Nest"]

ams_nest_2023.index = pd.to_datetime(
    ams_nest_2023.index.str.split(" ").str[0].str.replace("Z", "", regex=False),
    format="%Y-%m-%dT%H:%M:%S"
)

ams_nest_2023.head()

,building_name,electrical_energy,hot_water_energy
ts,,,
2023-01-01 00:00:00,AMS Nest,206.5,90.0
2023-01-01 01:00:00,AMS Nest,211.5,80.0
2023-01-01 02:00:00,AMS Nest,205.5,90.0
2023-01-01 03:00:00,AMS Nest,203.75,100.0
2023-01-01 04:00:00,AMS Nest,217.0,150.0


In [14]:
ams_2023_weather = pd.merge(weather_2023, ams_nest_2023, on="ts", how="inner")
ams_2023_weather = ams_2023_weather.drop(columns=["building_name", "snow_depth"])

ams_2023_weather.head()

,temperature_2m,wind_speed_10m,is_day,precipitation,relative_humidity_2m,cloud_cover,apparent_temperature,electrical_energy,hot_water_energy
ts,,,,,,,,,
2023-01-01 00:00:00,7.30,9.178235,1.0,0.0,87.0,12.0,4.803664,206.5,90.0
2023-01-01 01:00:00,7.15,2.902413,0.0,0.0,88.0,0.0,5.572490,211.5,80.0
2023-01-01 02:00:00,6.95,7.993297,0.0,0.0,94.0,8.0,4.796128,205.5,90.0
2023-01-01 03:00:00,6.90,13.217443,0.0,0.0,93.0,2.0,3.938524,203.75,100.0
2023-01-01 04:00:00,6.80,13.779114,0.0,0.0,89.0,4.0,3.597236,217.0,150.0


In [38]:
fig_07 = create_yearly_heatmap(ams_2023_weather, 2023, 1, 1, 31, 12, "WT2 Entire Year")

fig_07.update_layout(width=700, height=600)
fig_07.show()

(8734, 9)


Similar to the annual line charts for electrical and hot water usage, the overall correlation matrix does not reveal any strong linear relationships or immediately noticeable patterns, as many of the correlation values are close to 0.00. In particular, the relationship between electrical energy usage and hot water usage appears relatively weak when examining the aggregated yearly data.

One possible explanation is that temporal factors, such as season and month, influence these relationships and obscure correlations when the data is analyzed across an entire year. As a result, the relationships between variables may vary depending on the time period being observed. To investigate this further, we generate separate correlation matrices for January 2023 and December 2023 in order to examine whether temporal context affects the observed correlations.

In [54]:
fig_08 = make_subplots(rows=1, cols=2,
                        subplot_titles=["Janurary", "December"])

fig_08a = create_yearly_heatmap(ams_2023_weather, 2023, 5, 1, 31, 1, "Janurary")
fig_08b = create_yearly_heatmap(ams_2023_weather, 2023, 1, 12, 31, 12, "December")

# Assign separate color axes
fig_08a.data[0].update(coloraxis="coloraxis1")
fig_08b.data[0].update(coloraxis="coloraxis2")

fig_08.add_trace(fig_08a.data[0], row=1, col=1)
fig_08.add_trace(fig_08b.data[0], row=1, col=2)

fig_08.update_layout(
    width=1600, height=600,
    title_text="Weather vs Energy Correlation Heatmap 2023 for AMS Nest",
    coloraxis1=dict(colorscale='RdBu_r', cmin=-1, cmax=1),
    coloraxis2=dict(colorscale='RdBu_r', cmin=-1, cmax=1)
)

fig_08.show()

(623, 9)
(719, 9)


Based on the correlation heatmaps for January and December 2023, it becomes more apparent that accounting for temporal features reveals stronger and more interpretable relationships between variables. Compared to the yearly correlation matrix, fewer correlation values are close to zero, suggesting that the relationships between energy usage and other variables vary depending on the time period being analyzed.

These findings indicate that temporal features, such as month, season, and time of year, should be incorporated into predictive models in order to improve their accuracy and better capture context-dependent patterns in the data.

However, even with temporal information included, predictive models trained on historical data may still struggle to generalize to unforeseen or novel events that were not represented in the training data. Examples may include extreme weather conditions, campus closures, operational policy changes, or other irregular events that significantly alter building occupancy and energy usage patterns.